# Harmonized K=7 Neighborhood NMF robustness report

**Scope.** This report freezes the disease-blind evidence supporting a common coarse K=7 representation for K8, J2, and L3. J2/L3 K=12 models are deliberately deferred and are not analyzed here.

**Render command.** Run `bash scripts/1148_render_reference18_k7_robustness_report.sh` from the project root; the wrapper renders in a detached `tmux` session and writes a timestamped HTML file and log.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

root = Path.cwd().resolve()
while not (root / '.git').exists() and root != root.parent:
    root = root.parent
if not (root / '.git').exists():
    raise RuntimeError('Could not locate project root containing .git')

rank_run = root / 'results/reanalysis/REFERENCE18_NEIGHBORHOOD_NMF_K2_K12_RANK_SELECTION_20261016_110753'
primary_run = root / 'results/reanalysis/REFERENCE18_NEIGHBORHOOD_NMF_K7_K12_EXTENSION_20261015_attempt5'
sensitivity_run = root / 'results/reanalysis/REFERENCE18_NEIGHBORHOOD_NMF_K7_STATE_SENSITIVITY_20261016_210302_attempt5'
review_run = root / 'results/reanalysis/REFERENCE18_NEIGHBORHOOD_NMF_K7_SENSITIVITY_EVIDENCE_REVIEW_20261016_212300'
convergence_run = root / 'results/reanalysis/REFERENCE18_NEIGHBORHOOD_NMF_K8_K7_CONVERGENCE_EXTENSION_20261016_212100'
figure_run = root / 'results/reanalysis/REFERENCE18_K7_ROBUSTNESS_SUMMARY_20261017_165039'

rank = pd.read_csv(rank_run / '01_K2_K12_RANK_AUDIT.tsv', sep='\t')
model_robustness = pd.read_csv(sensitivity_run / '02_MODEL_ROBUSTNESS.tsv', sep='\t')
cross_chip = pd.read_csv(sensitivity_run / '07_CROSS_CHIP_FAMILY_MAP.tsv', sep='\t')
retained = pd.read_csv(review_run / '02_RETAINED_FAMILY_SUMMARY.tsv', sep='\t')
b_summary = pd.read_csv(review_run / '03_B_SUCCESSOR_SUMMARY.tsv', sep='\t')
decisions = pd.read_csv(review_run / '04_DECISIONS.tsv', sep='\t')
convergence = pd.read_csv(convergence_run / '01_CONVERGENCE_EXTENSION.tsv', sep='\t')
frozen = pd.read_csv(figure_run / '01_FROZEN_K7_INTERPRETATION.tsv', sep='\t')
validation = pd.read_csv(figure_run / '02_VALIDATION.tsv', sep='\t')
interpretation_amendment = pd.read_csv(figure_run / '03_INTERPRETATION_AMENDMENT_20261017.tsv', sep='\t')

assert decisions['pass'].all()
assert validation['pass'].all()
print(f'Project root: {root}')
print('All evidence-review and final-figure validations passed.')

## Methods and locked design

The primary models use the frozen 18-state Reference18 composition assay in centre-excluded 50-µm neighborhoods. NMF was fitted separately per chip with 495 balanced observations per patient, 50 random restarts, an unpenalized Frobenius objective, and patient leave-one-out checks. Disease labels were not supplied during fitting, rank review, factor matching, or sensitivity analysis.

K=7 is used as a harmonized coarse resolution, not claimed to be the independently optimal maximum resolution for every chip. The present scope intentionally excludes the J2/L3 K=12 models.

In [ ]:
display(frozen.style.hide(axis='index'))
display(interpretation_amendment.style.hide(axis='index'))

## Main result

The common K=7 solution contains seven cross-chip composition families. All matched loading cosines exceed the prespecified 0.80 threshold.

In [ ]:
main_cross = cross_chip.loc[cross_chip['scenario'].eq('main18')].copy()
main_cross['minimum_pairwise_cosine'] = main_cross[['K8_J2_cosine', 'K8_L3_cosine', 'J2_L3_cosine']].min(axis=1)
display(main_cross[['family_label', 'K8_factor', 'J2_factor', 'L3_factor', 'K8_J2_cosine', 'K8_L3_cosine', 'J2_L3_cosine', 'minimum_pairwise_cosine']].style.format(precision=3).hide(axis='index'))
display(Markdown(f"**Observed minimum cross-chip cosine:** {main_cross['minimum_pairwise_cosine'].min():.3f}."))

In [ ]:
display(Image(filename=str(figure_run / 'REFERENCE18_K7_ROBUSTNESS_SUMMARY.png'), width=1400))

## State-removal sensitivity

Two independent K=7 refits removed either Basal-like epithelial plus Lymphatic endothelial, or B. Rows were renormalized after state removal; all other fitting choices remained fixed. Independent best matches are used for retained-family review so that disappearance of one axis cannot force an artificial one-to-one reassignment.

In [ ]:
retained_display = retained.rename(columns={
    'minimum_best_loading_cosine': 'min_loading_cosine',
    'minimum_patient_usage_spearman': 'min_patient_score_rho',
    'minimum_q90_hotspot_jaccard': 'min_q90_jaccard',
})
display(retained_display[['scenario', 'chip', 'retained_family_n', 'min_loading_cosine', 'min_patient_score_rho', 'min_q90_jaccard']].style.format(precision=3).hide(axis='index'))
display(Markdown(
    f"Across all retained families, the minimum independent loading cosine was **{retained['minimum_best_loading_cosine'].min():.3f}**, "
    f"the minimum median-patient score Spearman was **{retained['minimum_patient_usage_spearman'].min():.3f}**, "
    f"and the minimum median-patient Q90 hotspot Jaccard was **{retained['minimum_q90_hotspot_jaccard'].min():.3f}**."
))

## B-enriched factor interpretation

After removing B, no single replacement factor coherently inherited the original B-enriched factor's patient-level score pattern and spatial hotspots. This does not mean the removed mass is unused: reconstruction necessarily redistributes it. It means that the original spatial factor is not simply renamed as one T/NK factor. The appropriate primary label is therefore **B-enriched/mixed-immune**, not pure B-cell abundance. Because the study is immune-focused, this mixed identity is biologically relevant rather than disqualifying: the factor is retained as a **primary immune-focused neighborhood axis** for downstream spatial, patient-level, and molecular characterization.

In [ ]:
display(b_summary.style.format(precision=3).hide(axis='index'))

## Extended convergence check

Because the initial K8 16-state run had a low fraction of restarts meeting the strict 5,000-step stopping criterion, the same 50 seeds were rerun with 20,000 allowed iterations. This distinguishes factor reproducibility from optimizer stopping behavior.

In [ ]:
display(convergence[['scenario', 'convergence_fraction', 'median_n_iter', 'maximum_n_iter', 'restart_fraction_all_factors_ge_0_80', 'matched_5k_to_20k_min_cosine']].style.format(precision=4).hide(axis='index'))

## Frozen interpretation and limitations

1. Use K=7 as the main cross-chip coarse representation.
2. Do not interpret the absence of separate T, NK, Mast, Basal, or AT2 factors at K=7 as absence of those signals. Their Reference18 weights participate in mixed factors, and a T-associated axis emerges when a dominant state is removed.
3. Treat Blood-endothelial, AT1-associated, Perivascular-stromal, Ciliated-associated, and Fibroblast-associated as the most robust core axes.
4. Prioritize B-enriched/mixed-immune as the main immune-focused neighborhood axis, with independent marker corroboration; do not equate its score with pure B-cell abundance.
5. Treat LEC-associated as sensitivity-qualified because upstream RCTD marker support was weak.
6. The composition signature defines the NMF factor. Gene-expression analysis is independent biological corroboration, not a prerequisite for fitting the RCTD-composition NMF.
7. No disease association is tested in this report.

In [ ]:
display(decisions.style.hide(axis='index'))

## Session information

In [ ]:
import session_info2
session_info2.session_info(dependencies=True, gpu=True)